# Linear

In [1]:
import torch
import torch.nn as nn

# Small example
seq_len = 3
embed_dim = 2
batch_size = 1

# Create sample input
x = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]])  # Shape: [1, 3, 2]
print(f"Input shape: {x.shape}")

# Linear layer connecting every position to every position
class FullyConnectedPositions(nn.Module):
  def __init__(self, seq_len, embed_dim):
    super().__init__()
    # Flatten sequence positions and embed dimensions together
    self.linear = nn.Linear(seq_len * embed_dim, seq_len * embed_dim)
    self.seq_len = seq_len
    self.embed_dim = embed_dim
      
  def forward(self, x):
    batch_size = x.shape[0]
    # Reshape to connect all positions: [batch, seq_len, embed] -> [batch, seq_len*embed]
    x_flat = x.reshape(batch_size, -1)
    # Apply linear layer
    output = self.linear(x_flat)
    # Reshape back
    return output.reshape(batch_size, self.seq_len, self.embed_dim)

# Create and apply model
full_model = FullyConnectedPositions(seq_len, embed_dim)
out_full = full_model(x)
print(f"Full model parameters: {sum(p.numel() for p in full_model.parameters())}")

Input shape: torch.Size([1, 3, 2])
Full model parameters: 42


## Batch Matrix Multiplication

`torch.einsum` can do what `torch.bmm` does. It's good to know multiple ways.

In [11]:
q = torch.arange(0,12).view(2, 2, 3)
print(f"q shape: {q.shape}")
print(f"q:\n{q}")
k = torch.arange(0,12).view(2, 2, 3)
print(f"k shape: {k.shape}")
print(f"k:\n{k}")
k_t = k.transpose(1, 2)
print(f"k^T shape: {k_t.shape}")
print(f"k^T:\n{k_t}")
res = torch.bmm(q, k.transpose(1, 2))
print(f"res shape: {res.shape}")
print(f"res:\n{res}")

q shape: torch.Size([2, 2, 3])
q:
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])
k shape: torch.Size([2, 2, 3])
k:
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])
k^T shape: torch.Size([2, 3, 2])
k^T:
tensor([[[ 0,  3],
         [ 1,  4],
         [ 2,  5]],

        [[ 6,  9],
         [ 7, 10],
         [ 8, 11]]])
res shape: torch.Size([2, 2, 2])
res:
tensor([[[  5,  14],
         [ 14,  50]],

        [[149, 212],
         [212, 302]]])


# Scaled Dot Product Attention

In [2]:
# Simple attention implementation
class SimpleAttention(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()
    self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
    self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
    self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
    self.out_proj = nn.Linear(embed_dim, embed_dim)
    self.embed_dim = embed_dim
      
  def forward(self, x):
    # Project to queries, keys, values
    q = self.q_proj(x)  # [batch, seq, embed]
    k = self.k_proj(x)  # [batch, seq, embed]
    v = self.v_proj(x)  # [batch, seq, embed]
    
    # Calculate attention scores
    scores = torch.bmm(q, k.transpose(1, 2)) / (self.embed_dim ** 0.5)  # [batch, seq, seq]
    
    # Apply softmax to get attention weights
    attn_weights = torch.softmax(scores, dim=-1)  # [batch, seq, seq]
    
    # Apply attention weights to values
    output = torch.bmm(attn_weights, v)  # [batch, seq, embed]
    
    return self.out_proj(output)

# Create and apply model
attn_model = SimpleAttention(embed_dim)
out_attn = attn_model(x)
print(f"Attention model parameters: {sum(p.numel() for p in attn_model.parameters())}")

# Show attention weights for visualization
with torch.no_grad():
  q = attn_model.q_proj(x)
  k = attn_model.k_proj(x)
  scores = torch.bmm(q, k.transpose(1, 2)) / (embed_dim ** 0.5)
  attn_weights = torch.softmax(scores, dim=-1)
  print("Attention weights (connecting all positions):")
  print(attn_weights[0])

Attention model parameters: 18
Attention weights (connecting all positions):
tensor([[7.6768e-01, 1.8684e-01, 4.5474e-02],
        [9.6184e-01, 3.6755e-02, 1.4045e-03],
        [9.9400e-01, 5.9637e-03, 3.5781e-05]])
